In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [2]:
columns = ["age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
           "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
           "hours-per-week", "native-country", "income"]

df = pd.read_csv("../data/raw/adult.data", names=columns, na_values=" ?", skipinitialspace=True)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

df.head()


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [3]:
# Encodage binaire de la cible
df["income"] = df["income"].apply(lambda x: 1 if x == ">50K" else 0)

# Encodage des colonnes catégorielles
categorical_cols = df.select_dtypes(include="object").columns.drop("income")
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

df_encoded.head()

KeyError: "['income'] not found in axis"

In [ ]:
df.describe(include="all")

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df_encoded.corr()[["income"]].sort_values(by="income", ascending=False), annot=True, cmap="coolwarm")
plt.title("Corrélation avec le revenu (>50K)")
plt.show()

In [ ]:
features = ["age", "education-num", "hours-per-week", "capital-gain", "capital-loss"]
X = df[features]
X_scaled = StandardScaler().fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=0)
df["cluster"] = kmeans.fit_predict(X_scaled)

sns.pairplot(df, vars=features, hue="cluster", palette="tab10")
plt.show()


In [ ]:
X = df_encoded.drop("income", axis=1)
y = df_encoded["income"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
